# This is a file to prepare the LLM Manifest json file to make an LLM version of the file

In [1]:
import json # for handling all the json files (loading and saving files)
import ijson # For large json files. Allows for streaming of data. (I don't think this is used often since most files are small.)
import os  # To move through the file tree to get to the .env file with the API Key
import re # Regular expression for matching and finding terms to highlight.
import copy # For making deep copies of arrays and objects.
from dotenv import load_dotenv  # Keeping API secrets
from openai import OpenAI, BadRequestError  # Connect to LLM (ChatGPT)
from fuzzywuzzy import process # for highlighting terms that are similar in capitalization and such.

load_dotenv() # load the api key into memory.

True

In [2]:
# Define the Conditions for this run.
useGPT = True

colors = {
    "Doc_open": "crimson",
    "Documents Opened": "crimson",
    "Topics": "0096FF",
    "Search": "#009420",
    "Searches": "#009420",
    "Add note": "#4278f5",
    "Notes": "#4278f5",
    "Highlight": "#ab8300",
    "Highlights": "#ab8300",
    "Reading": "pink",
    "Keywords": "#0096FF",
    "Dates": "#b16a1f",
    "PersonEnt": "#0096FF",
    "GeoEnt": "#049c9a",
    "barBG": "lightgrey",
    "Average-neg": "blue",
    "Average-pos": "orange",
}

parentDirectory = os.path.join(
    os.path.abspath(os.path.join(os.getcwd(), os.pardir)), os.getcwd()
)
parentDirectory += "/.."

totalSegmentsCnt = 6
datasetCnt = 3
participantCnt = 8
dataset_path = parentDirectory + f"/interface/ApplicationManifest_{totalSegmentsCnt}.json"
LLMPartialsDirectory = parentDirectory + f"/data/LLMStages_{totalSegmentsCnt}/"

In [3]:
def load_json_file_old(file_path):
    with open(file_path, "r") as read_file:
        data = json.load(read_file)
    return data


def load_json_file(file_path, threshold_mb=5):
    file_size = os.path.getsize(file_path) / (1024 * 1024)  # Convert bytes to megabytes

    try:
        if file_size <= threshold_mb:
            # For small files, use json.load
            with open(file_path, "r") as read_file:
                data = json.load(read_file)
            return data
        else:
            # For large files, use ijson to parse
            data = []
            with open(file_path, "r") as read_file:
                parser = ijson.parse(read_file)
                for prefix, event, value in parser:
                    data.append((prefix, event, value))
            return data
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
    except json.JSONDecodeError:
        print(f"Error: The file {file_path} is not a valid JSON file.")
    except MemoryError:
        print("Error: The file is too large to load into memory.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


def save_json_to_file(data, filename, overwriteFiles=True, indent=True):
    """
    Save a JSON object to a file.

    Parameters:
    data (dict): The JSON object to save.
    filename (str): The name of the file to save the JSON object in.
    overwriteFiles (bool): Optional - if you are done testing the files and don't want to overwrite files already made, flip this False.
    """
    # Create the directory if it doesn't exist
    if overwriteFiles:
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        with open(filename, "w") as file:
            if indent:
                json.dump(data, file, indent=2)
            else:
                json.dump(data, file, separators=(",", ":"))
    else:
        # Make a file with test as the predicate so not to overwrite the version you like
        filename = filename[:-5] + "-test.json"
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        with open(filename, "w") as file:
            if indent:
                json.dump(data, file, indent=2)
            else:
                json.dump(data, file, separators=(",", ":"))


def merge_json(data1, data2):
    # Create a dictionary to store the merged data
    merged_data = {}

    # Add all entries from the first file to the merged_data
    for entry in data1:
        merged_data[entry["id"]] = entry

    # Merge entries from the second file
    for entry in data2:
        if entry["id"] in merged_data:
            # If the 'id' already exists, update the existing entry with new data
            merged_data[entry["id"]].update(entry)
        else:
            # Otherwise, add the new entry
            merged_data[entry["id"]] = entry

    # Convert the merged_data back to a list of objects
    # merged_list = list(merged_data.values())
    # return merged_list
    return merged_data

In [12]:
def checkEventType (event, disqualifiedEvents = ["Think_aloud", "Topic_change", "Mouse_hover"]):
    if event["interactionType"] in disqualifiedEvents:
        return False
    else:
        return True

def remove_properties(obj, properties):
    """
    Removes specified properties from a dictionary if they exist, 
    or if the properties have empty string or empty list values.

    Parameters:
    obj (dict): The dictionary object from which properties are to be removed.
    properties (list): A list of property names (keys) to be removed from the dictionary.

    Returns:
    dict: The dictionary object with the specified properties removed.
    """
    for prop in properties:
        if prop in obj:
            del obj[prop]
    return obj

def return_blank_properties(obj):
    """
    Removes properties from a dictionary if they have empty string or empty list values.

    Parameters:
    obj (dict): The dictionary object from which properties with empty values are to be removed.

    Returns:
    dict: The dictionary object with properties having empty string or empty list values removed.
    """
    props = list(obj.keys())  # Create a list of keys to avoid changing the dictionary size during iteration
    for p in props:
        if obj[p] == '' or obj[p] == []:
            del obj[p]
    return obj

def addContextToInteraction(interaction, allDocuments, allowable_types=["Reading","Doc_open","Highlight"], propertiesToAppend = ["summary","Geos","People","topics"], propertiesToRemove = ["id", "duration", "dataset", "PID", "segment"]):
    # Check if the interactionType is in the allowable types
    if interaction.get('interactionType') in allowable_types:
        # Find the corresponding object in the larger object using the 'id' property
        matchingID = interaction.get('id')
        dataToAdd = allDocuments.get(matchingID)
        

        # If a matching object is found, update the obj with its information
        if dataToAdd:
            update_data = {k: dataToAdd[k] for k in propertiesToAppend if k in dataToAdd}
            interaction.update(update_data)

    interaction = remove_properties(interaction, propertiesToRemove)
    interaction = return_blank_properties(interaction)
    
    return interaction


def askGPTMultiple(prompt):
    print("✨ too long ❌ - asking in chunks")

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    # Determine max token length (adjust if necessary)
    max_token_length = 14000
    response_token_length = 150  # Expected token length of each response

    # Split prompt into segments that fit within max token length
    chunks = []
    while len(prompt) > max_token_length:
        split_index = prompt.rfind(' ', 0, max_token_length)  # find a suitable split point
        if split_index == -1:
            break
        chunks.append(prompt[:split_index+1])
        prompt = prompt[split_index+1:] #get the portion of the string remaining
    chunks.append(prompt) #add remaining string

    sys_msg = [{"role": "system", "content": "Summarize as concisely as possible each chunk of interactions I give you. Update the summary to incorporate each chunk into your summary."}]
    # Initialize with an empty assistant response
    assistant_response = ""

    # Send each segment to the API
    print("preparing message")
    for chunk in range(len(chunks)):
        print(f"message {chunk+1}/{len(chunks)} is {len(chunks[chunk])} characters")

        # Construct the current message set
        current_msg = sys_msg + [{"role":"assistant", "content":assistant_response}, {"role":"user", "content":chunks[chunk]}]

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=current_msg, # type: ignore
            max_tokens=response_token_length,  # Adjust based on the desired summary length
            temperature=0.1,  # Adjust for summary precision
        )

        # Update the assistant response for the next segment
        assistant_response = response.choices[0].message.content
        print("assistant says:", assistant_response)
        # msg.append({"role": "assistant", "content": assistant_response}) # type: ignore

    return assistant_response


def askGPTMultiple_broken(prompt):
    print("✨ too long ❌ - asking in chunks")

    client = OpenAI(
        # This is the default and can be omitted
        api_key=os.environ.get("OPENAI_API_KEY"),
    )
    # Determine max token length (typically 2048 tokens)
    max_token_length = 10000

    # Split prompt into segments that fit within max token length
    chunks = []
    while len(prompt) > max_token_length:
        split_index = prompt.rfind('}', 0, max_token_length)  # find a suitable split point
        if split_index == -1:
            break
        chunks.append(prompt[:split_index+1])
        prompt = prompt[split_index+1:] #get the portion of the string remaining
    chunks.append(prompt) #add remaining string

    msg = [{"role": "system", "content": "Summarize as concisely as possible each chunk of interactions I give you. Update the summary to incorporate each chunk into your summary."}]
    # Send each segment to the API
    print("preparing message")
    for chunk in range(len(chunks)):
        print(f"message {chunk}/{len(chunks)-1} is {len(chunks[chunk])} characters")
        msg.append({"role":"user", "content":chunks[chunk]})

        response = client.chat.completions.create(
            messages=msg, # type: ignore
            model="gpt-3.5-turbo",
            max_tokens=150,  # Adjust based on the desired summary length
            temperature=0.1,  # Adjust for summary precision
        )

        # Update context for the next segment
        current_response = response.choices[0].message.content
        msg.append({"role": "assistant", "content": current_response}) # type: ignore
    return msg[-1]["content"]


def askGPT(prompt):
    print(f"asking ✨GPT: {prompt[:400]}...")
    try:
        # Create a prompt that asks the model to summarize the logs

        client = OpenAI(
            # This is the default and can be omitted
            api_key=os.environ.get("OPENAI_API_KEY"),
        )
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="gpt-3.5-turbo",
            max_tokens=150,  # Adjust based on the desired summary length
            temperature=0.1,  # Adjust for summary precision
        )

        return chat_completion.choices[0].message.content
    except BadRequestError as e:
        print(str(e))
        # if 'error' in e and 'code' in e['error'] and e['error']['code'] == 'context_length_exceeded':
        return askGPTMultiple(prompt)
            # return "Error: Maximum context length exceeded. Please reduce the length of your prompt or use a shorter prompt."
        # else:
        #     return f"Error: {str(e)}"
    except Exception as e:
        return f"Error: {str(e)}"

"""
Finalizes the dataset processing by cleaning up interactions, summarizing segments,
identifying entities, and saving the summaries.

This function performs the following steps:
1. Cleans up each interaction to reduce the token count.
2. Summarizes each segment's interactions using ChatGPT.
3. Identifies entities in each segment summary and overall summary.
4. Parses summaries and adds HTML code to highlight entities.
5. Saves the summaries to files for each user.
"""
def clean_interaction(interaction):
    """
    Cleans an interaction by removing unnecessary parts.

    Parameters:
    interaction (dict): The interaction object to clean.

    Returns:
    dict: The cleaned interaction object.
    """
    keys_to_keep = ['id', 'interactionType', 'time', 'segment', 'summary', 'Geos', 'People', 'topics']
    return {key: interaction[key] for key in keys_to_keep if key in interaction}

def summarize_segment(segment_interactions):
    """
    Summarizes interactions in a segment using ChatGPT.

    Parameters:
    segment_interactions (list): List of interactions in the segment.

    Returns:
    str: The summary of the segment.
    """
    interactions_text = "\n".join([str(interaction) for interaction in segment_interactions])
    prompt = f"Here is a list of interactions from a user's investigation of data. Please provide a concise 500-character explanation focusing on main topics and findings. Include actionable instructions, relevant keywords, and ensure accuracy and completeness. Highlight unique or unusual interactions from this set of user's data analysis, using specific words since this summary will be incorporated with a broader report on the work an individual did. These interactions are only a segment of the whole investigation:\n\n{interactions_text}"
    # print("🌕 summarize segment:", prompt)
    if useGPT:
        return askGPT(prompt)
    else:
        return "This would be a summary. The words are easy to read with nice things to say. You May not enjoy it all but at least it is a nice length."

def summarize_all_segments(segment_summaries):
    """
    Summarizes interactions in a segment using ChatGPT.

    Parameters:
    segment_interactions (list): List of interactions in the segment.

    Returns:
    str: The summary of the segment.
    """
    interactions_text = "\n".join([str(summary) for summary in segment_summaries])
    prompt = f"Below are {totalSegmentsCnt} short descriptions from segments of a user's investigation of a particular dataset. Please generate an executive summary about the users investigation. Your summary will be a maximum of 500 characters and explain the main events, topics covered, and findings. Summarize the following:\n{interactions_text}"
    # print("🌕 summarize segment:", prompt)
    if useGPT:
        return askGPT(prompt)
    else:
        return "Overarching summary is now a bit longer and interesting but it refers to places like segment 3 and such."

def update_entities(entities, key, value):
    # Convert the key to lowercase for case-insensitive matching
    key = key.lower()
    
    # Mapping of keys to entity types
    # These mappings define how different keys should update the 'entities' dictionary
    cases = {
        "search": "Search",             # Maps "search" and "searches" to "Search"
        "searches": "Search",
        "topics": "Doc_open",           # Maps "topics" to "Doc_open"
        "concepts": "Highlights",       # Maps "concepts" to "Highlights"
        "people": "PersonEnt",          # Maps "people" and "persons" to "PersonEnt"
        "persons": "PersonEnt",
        "places": "GeoEnt",             # Maps "places" and "locations" to "GeoEnt"
        "locations": "GeoEnt",
        "dates": "Dates",               # Maps "dates" to "Dates"
        "document titles": "Doc_open"   # Maps "document titles" to "Doc_open"
    }

    # Check if the key is in the predefined mapping
    if key in cases:
        entity_key = cases[key]
        
        # Ensure value is always a list, even if it's a single value
        if not isinstance(value, list):
            value = [value]

        # Update entities dictionary based on entity_key
        if entity_key in entities:
            if isinstance(entities[entity_key], list):
                # If entity_key already exists and is a list, extend it with new values
                entities[entity_key].extend(value)
            else:
                # If entity_key exists but is not a list, convert it to a list and add new values
                entities[entity_key] = [entities[entity_key]] + value
        else:
            # If entity_key does not exist in entities, initialize it with the new values
            entities[entity_key] = value


def identify_entities(summary, attempts = 0, maxAttempts = 5):
    """
    Identifies entities in a summary.

    Parameters:
    summary (str): The summary text.

    Returns:
    dict: A dictionary of identified entities.
    """
    # Step 1: Construct a prompt
    # prompt = f"format your reply as a JSON object. Identify and list any of the 6 types of entities in the following text:\n{summary}\nEntities to look for: [search, topics, people, places, dates, document titles]. Make a dictionary with each of these Entities as keys and list any (or none) of the terms as values."

    prompt = f"Identify and make a short list of the following 4 kinds of information in the attached summary: [searches, highlights, people, places]. There should be less than 5 words in each list. Make a dictionary with each entity type as keys and lists of exact terms as the values. If there is nothing applicable, you can provide an empty list for that entity type (e.g.,  ['']). Your response is valid JSON and properly escapes any necessary characters. Prioritize the 5 most prominent and specific searches, highlights, people, and places the individual read about in the following summary:\n\n{summary}"

    default_entities = {
        "Search": [""],
        "Highlights": [""],
        "PersonEnt": [""],
        "GeoEnt": [""],
    }

    # Step 2: Use askGPT to get the response
    entities = {}
    if(useGPT):
        try:
            response = askGPT(prompt)
            # Attempt to repair common JSON issues
            response = response.replace('\n', ' ').replace('\r', '')  # type: ignore
            structured_response = json.loads(response) #type: ignore
            print(structured_response, f"\033[32m{type(structured_response)}\033[0m")
            # Step 3: Parse the response to extract entities
            for key, value in structured_response.items():
                update_entities(entities, key, value)
        except TypeError as e:
            print(f"Uh oh: {e}")
        except json.decoder.JSONDecodeError as f:
            entities = default_entities            
            #try again so long as it's not too many attempts.
            # if attempts < maxAttempts :
            print(f"decode error This is not JSON - \033[31m{f}\033[0m::::{response}")
            #     attempts += 1
            #     entities = identify_entities(summary, attempts, maxAttempts)
            # else:

    else:
        entities = default_entities

    return entities

def fuzzy_highlight_entities(summary, entities, threshold=80):
    """
    Highlights entities in the summary text with HTML code.

    Parameters:
    summary (str): The summary text.
    entities (dict): A dictionary of entities to highlight.

    Returns:
    str: The summary text with highlighted entities.
    """
    words = summary.split()

    for entity_type, terms in entities.items():
        color = colors[entity_type]
        for term in terms:
            newTerm = str(term).replace(" ", "_").lower()

            # Find the best matches for the term in the summary
            matches = process.extract(term, words, limit=len(words))
            
            for match, score in matches: # type: ignore
                if score >= threshold:
                    highlighted_term = f"<span onmouseover=highlightSimilar('{newTerm}') onmouseout=unhighlightCards() class='descriptionTerm {entity_type}' style='color:{color}; font-weight:bold'>{match}</span>"
                    summary = summary.replace(match, highlighted_term)
    return summary

def highlight_entities_memory_overflow_bad(summary, entities):
    """
    Highlights entities in the summary text with HTML code.

    Parameters:
    summary (str): The summary text.
    entities (dict): A dictionary of entities to highlight.

    Returns:
    str: The summary text with highlighted entities.
    """
    for entity_type, terms in entities.items():
        print(entity_type, terms)
        color = colors[entity_type]
        for term in terms:
            noSpaceTerm = str(term).replace(" ", "_").lower()
            summary = summary.replace(term, f"<span onmouseover=highlightSimilar('" + noSpaceTerm +"" + "') onmouseout=unhighlightCards() class='descriptionTerm "+entity_type+"' style='color:" + color + "; font-weight:bold' >" + term + "</span>")
    return summary

def highlight_entities(summary, entities):
    """
    Highlights entities in the summary text with HTML code.

    Parameters:
    summary (str): The summary text.
    entities (dict): A dictionary of entities to highlight.

    Returns:
    str: The summary text with highlighted entities.
    """
    def replacement(match):
        term = match.group(0)
        noSpaceTerm = term.replace(" ", "_").lower()
        entity_type = term_to_entity[term]
        color = colors[entity_type]
        return f"<span onmouseover=highlightSimilar('{noSpaceTerm}') onmouseout=unhighlightCards() class='descriptionTerm {entity_type}' style='color:{color}; font-weight:bold'>{term}</span>"

    # Create a mapping from terms to their entity types
    term_to_entity = {}
    patterns = []
    for entity_type, terms in entities.items():
        for term in terms:
            if(term != ""): #so long as the term is not empty, add it to the list of replacements
                term_to_entity[term] = entity_type
                patterns.append(r'\b' + re.escape(term) + r'\b')

    # Build a regex pattern to match all terms
    combined_pattern = re.compile('|'.join(patterns))

    # Replace terms with highlighted HTML
    highlighted_summary = re.sub(combined_pattern, replacement, summary)

    return highlighted_summary
   
def highlight_entities_case_insensitive(summary, entities):
    """
    Highlights entities in the summary text with HTML code.

    Parameters:
    summary (str): The summary text.
    entities (dict): A dictionary of entities to highlight.

    Returns:
    str: The summary text with highlighted entities.
    """
    def replace_with_highlight(match, newTerm, entity_type, color):
        original_term = match.group(0)
        return f"<span onmouseover=highlightSimilar('{newTerm}') onmouseout=unhighlightCards() class='descriptionTerm {entity_type}' style='color:{color}; font-weight:bold' >{original_term}</span>"
    
    for entity_type, terms in entities.items():
        color = colors[entity_type]
        for term in terms:
            newTerm = str(term).replace(" ", "_").lower()
            pattern = re.compile(re.escape(term), re.IGNORECASE)
            summary = pattern.sub(lambda match: replace_with_highlight(match, newTerm, entity_type, color), summary)
    
    return summary

def replace_with_dict(text, replacements):
    # Iterate through the dictionary
    for key, value in replacements.items():
        # Create a regex pattern to find the key
        pattern = re.compile(re.escape(key))
        # Replace the key with the value in the text
        text = pattern.sub(value, text)
    return text

def assignSegment(interaction, ds, pid, segments, debug = False):
    #try to access the 'segment' property. Return the interaction if the property exists
    try:
        int(interaction["segment"])
        return interaction
    #if no property, then attempt to identify the appropriate segment.
    except KeyError:
        if debug: print("Found an event without a 'segment' property, trying to assign appropriate segment:")
        tCheck = int(interaction["time"])
        for i in range(totalSegmentsCnt):
            start = segments[ds+pid+i]["start"]
            end = segments[ds+pid+i]["end"]
            if debug: print("--",interaction["interactionType"], start, tCheck, end)
            if start <= tCheck <= end:
                if debug: print(f"--> Found the correct segment number for interaction '{interaction["type"]}' at {interaction["time"]}")
                interaction.update({"segment":i})
                break
            elif (i + 1 == totalSegmentsCnt):
                if debug: print("assigning the last segment for the interaction event:", totalSegmentsCnt-1)
                interaction["segment"] = totalSegmentsCnt-1
    return interaction

In [5]:
def print_json_shape(data, max_depth=None, level=0, priorKey=None):
    indent = "  " * level
    if max_depth is not None and level >= max_depth:
        print(f"{indent}...")
        return

    if isinstance(data, dict):
        print(f'{indent}"{priorKey}": Object with {len(list(data.keys()))} keys: {list(data.keys())}')
        for key, value in data.items():
            # print(f"{indent}  Key: {key}")
            print_json_shape(value, max_depth, level + 1, key)
    elif isinstance(data, list):
        print(f"{indent}\"{priorKey}\": Array with {len(data)} items")
        for index, item in enumerate(data):
            print_json_shape(item, max_depth, level + 1, "array")
            if max_depth is not None and level + 1 >= max_depth:
                break
    # else:
    # print(f"{indent}Value: {data}")
# Example usage
json_data = """
{
  "name": "John",
  "age": 30,
  "children": [
    {
      "name": "Jane",
      "age": 10
    },
    {
      "name": "Doe",
      "age": 5
    }
  ],
  "address": {
    "street": "123 Main St",
    "city": "Anytown"
  }
}
"""

# Parse the JSON string into a Python object
parsed_data = json.loads(json_data)

# Print the shape of the JSON object
print_json_shape(parsed_data)

"None": Object with 4 keys: ['name', 'age', 'children', 'address']
  "children": Array with 2 items
    "array": Object with 2 keys: ['name', 'age']
    "array": Object with 2 keys: ['name', 'age']
  "address": Object with 2 keys: ['street', 'city']


## Load some data and set up the other functions.

In [ ]:
# 1. Read the dataset into memory
data = load_json_file(dataset_path)
print(data.keys()) # type: ignore
# print_json_shape(data,0,3)

interaction_logs = data["interactionLogs"]  # type: ignore # keep this line for when you bring loops back in.
segments = data["segments"] # type: ignore
# print_json_shape(segments,0, 2)

## Step 1; ✨ summarize the documents

In [ ]:

# Iterate over each document dataset
for user in range(0,datasetCnt):
# if True:
    # user = 0
    documents_path = parentDirectory+"/data/Dataset_"+str(user+1)+"/Documents/Documents_Dataset_"+str(user+1)+".json"
    print("🚀 ~ documents_path:", documents_path)
    tempdocs = load_json_file(documents_path)
    docs = []

    LLMStage = "01-summary/"
    for doc in tempdocs:
        if useGPT:
            summaryString = askGPT('Please examine the content of this JSON record record and provide a 1 sentence summary of the content and a short list of the topics. Your response should have no additional characters or padding. return a json object with the form: {"summary":"Generate summary text here.","topics":["topic1","topic2","topic3"]}. Here is the record:\n\n'+str(doc))
        else:
            summaryString = '{"Summary":"This would be a summary from GPT", "topics":["one","two","three"]}'
        # Parse JSON string to dictionary
        summary = json.loads(str(summaryString))
        print(f"doc:{type(doc)} | summary is --- {summary}")
        doc.update(summary)
        docs.append(doc)
    save_json_to_file(docs, LLMPartialsDirectory+LLMStage+"documents"+str(user)+".json")


## Step 2; Merge the entities with the document data

In [ ]:
# Iterate over interaction logs
allDocs = {}
LLMStage = "02-merging/"
for user in range(0,datasetCnt):
# if True:
#     user = 0
    entities_path = parentDirectory+"/data/Dataset_"+str(user+1)+"/Documents/Entities_Dataset_"+str(user+1)+".json"
    tempPeepPlace = load_json_file(entities_path)
    docs = load_json_file(LLMPartialsDirectory + "01-summary/documents" + str(user) + ".json")
    documents = merge_json(docs,tempPeepPlace)
    allDocs.update(documents)
    save_json_to_file(documents, LLMPartialsDirectory+LLMStage+"documents"+str(user)+".json")
    # documents = load_json_file(
    #     LLMPartialsDirectory + LLMStage + "documents" + str(user) + ".json"
    # )
save_json_to_file(allDocs, LLMPartialsDirectory + LLMStage+"allDocuments.json")

In [ ]:
print_json_shape(documents)

## Step 3; Assign interactions to a segment

In [ ]:
LLMStage = "03-segmentation/"
# Initialize a dictionary to hold interactions by segments
segment_interactions = {i: [] for i in range(len(segments))}
# Iterate over interaction logs
for user in range(len(interaction_logs)):
# if True:
    # user = 0
    # Iterate over each interaction of the user
    for interactions in interaction_logs[user]:
        tempset = []
        lastsegment = 0
        for interaction in interactions:
            ds = int(interaction["dataset"]) - 1
            pid = int(interaction["PID"]) - 1
            seg = assignSegment(interaction, ds, pid, segments)["segment"]
            #assign the interaction to a grouped set of interactions.
            
            if checkEventType(interaction):
                if seg == lastsegment:
                    tempset.append(interaction)
                else:
                    index = ds*participantCnt*totalSegmentsCnt + pid*totalSegmentsCnt + seg - 1
                    # print(f"This event belongs to seg {seg}, ds {ds}, and pid {pid}, moving to the next segment: {index}")
                    # segment_interactions[index].append(interaction)
                    segment_interactions.update({index:tempset})
                    lastsegment = seg
                    tempset = [interaction]
            # Ensure the final segment is captured (in case this is the last interaction)
            final_index = ds * participantCnt * totalSegmentsCnt + pid * totalSegmentsCnt + lastsegment
            segment_interactions.update({final_index: tempset})
save_json_to_file(segment_interactions, LLMPartialsDirectory + LLMStage + "segmented_interactions.json")
print_json_shape(segment_interactions,2)

In [6]:
segment_interactions = load_json_file(
    LLMPartialsDirectory + "03-segmentation/segmented_interactions.json"
)
documents = load_json_file(LLMPartialsDirectory + "02-merging/allDocuments.json")
segmented_context_interactions = {}


# print(segment_interactions.keys())
# print_json_shape(segment_interactions, 2)

# Clean up and Add Context to the interactions
for seg_idx in range(len(segment_interactions)): # type: ignore
    segmented_context_interactions[str(seg_idx)] = []
    for interaction in segment_interactions[str(seg_idx)]: # type: ignore
        # print(((interaction)))
        context_interaction = addContextToInteraction(interaction,documents)
        # print(((interaction)))
        segmented_context_interactions[str(seg_idx)].append(context_interaction)
    print(
        f"Cleaned up and contextualized {len(segmented_context_interactions[str(seg_idx)])} interactions in segment #{seg_idx}"
    )

LLMStage = "03-ContextInteractions/"
save_json_to_file(segmented_context_interactions,LLMPartialsDirectory+LLMStage+"Segmented_Context_interactions.json",True,False)
# print(len(segment_interactions))

Cleaned up and contextualized 116 interactions in segment #0
Cleaned up and contextualized 80 interactions in segment #1
Cleaned up and contextualized 66 interactions in segment #2
Cleaned up and contextualized 62 interactions in segment #3
Cleaned up and contextualized 89 interactions in segment #4
Cleaned up and contextualized 176 interactions in segment #5
Cleaned up and contextualized 103 interactions in segment #6
Cleaned up and contextualized 34 interactions in segment #7
Cleaned up and contextualized 51 interactions in segment #8
Cleaned up and contextualized 98 interactions in segment #9
Cleaned up and contextualized 58 interactions in segment #10
Cleaned up and contextualized 163 interactions in segment #11
Cleaned up and contextualized 106 interactions in segment #12
Cleaned up and contextualized 83 interactions in segment #13
Cleaned up and contextualized 98 interactions in segment #14
Cleaned up and contextualized 141 interactions in segment #15
Cleaned up and contextualize

## Part 4; ✨ Creating summary cards for each segment

First we need to generate segment summaries and get lists of entities in the summaries with ChatGPT.

In [7]:
# running the entire function takes about 17 minutes to complete
# segmented_context_interactions = load_json_file(
#     LLMPartialsDirectory
#     + "03-ContextInteractions/Segmented_Context_interactions.json"
# )

LLMStage = "/04-segmentSummaries/"
idx_to_start_from = 41
# idx_to_end_at = len(segmented_context_interactions)
idx_to_end_at = 47

specific_idx_to_fix = 44 # in case you need to run this again, here's a way to adjust which segment gets overwritten.

ask_gpt_for_summary = True
ask_gpt_for_entities = True
for seg_idx in range(idx_to_start_from, idx_to_end_at):
    if seg_idx == specific_idx_to_fix:
        # if True:
        print(f"---Summarizing segment_{seg_idx}")
        # have to use the string version of the index to access the value in dictionary. Integer version is not a valid key
        segment = segmented_context_interactions[str(seg_idx)] # type: ignore 
        cleaned_interactions = [clean_interaction(interaction) for interaction in segment]
        if ask_gpt_for_summary: 
            summary = summarize_segment(cleaned_interactions)
            save_json_to_file(summary, LLMPartialsDirectory + LLMStage + f"SegmentSummaries/segment_{seg_idx}.json")
        else: 
            summary = load_json_file(LLMPartialsDirectory + LLMStage + f"SegmentSummaries/segment_{seg_idx}.json")
        if ask_gpt_for_entities:
            entities = identify_entities(summary)
            save_json_to_file(entities, LLMPartialsDirectory + LLMStage + f"SegmentEntities/segment_{seg_idx}.json")

---Summarizing segment_44
asking ✨GPT: Here is a list of interactions from a user's investigation of data. Please provide a concise 500-character explanation focusing on main topics and findings. Include actionable instructions, relevant keywords, and ensure accuracy and completeness. Highlight unique or unusual interactions from this set of user's data analysis, using specific words since this summary will be incorporated with a broad...
Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 17462 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
✨ too long ❌ - asking in chunks
preparing message
message 1/5 is 13992 characters
assistant says: The user's investigation reveals arms dealing activities in various locations like Kenya, Nigeria, and Yemen. Intercepted communications and intelligence reports expose relationships

Then combine the entities and summaries into a single file and apply highlighting colors. Save both versions as single files.

In [19]:
# segmented_context_interactions = load_json_file(
# LLMPartialsDirectory + "03-ContextInteractions/Segmented_Context_interactions.json"
# )

print("Loaded Segment Context")
LLMStage = "/04-segmentSummaries/"
total_segments = participantCnt * totalSegmentsCnt * datasetCnt # len(segmented_context_interactions)
print("🚀 ~ total_segments:", total_segments, "| should be 144 for 6 segments.")
user_summaries = [""] * total_segments
raw_summaries = [""] * total_segments

for seg_idx in range(total_segments):
    print(f"🥣 ~ Mixing seg_idx {seg_idx} into one file")
    summary = load_json_file(LLMPartialsDirectory + LLMStage + "SegmentSummaries/segment_" + str(seg_idx) + ".json")
    entities = load_json_file(LLMPartialsDirectory + LLMStage + "SegmentEntities/segment_"+str(seg_idx)+".json")
    highlighted_summary = highlight_entities(summary, entities)
    replacements = {
        "Keywords: ": "<br /><strong>Keywords:</strong> ",
        "Actionable steps ": "<strong>Actionable steps</strong> ",
        "Actionable insights ": "<strong>Actionable insights</strong> ",
        "Key findings": "<strong>Key findings</strong> ",
        "Actionable instructions " : "<strong>Actionable instructions</strong> "
    }
    bolded_summary = replace_with_dict(highlighted_summary, replacements)

    raw_summaries[seg_idx] = summary # type: ignore
    user_summaries[seg_idx] = bolded_summary # type: ignore
save_json_to_file(raw_summaries, LLMPartialsDirectory + f"Raw_segment_summaries_{totalSegmentsCnt}.json")
save_json_to_file(user_summaries, LLMPartialsDirectory + f"Highlighted_segment_summaries_{totalSegmentsCnt}.json")

Loaded Segment Context
🚀 ~ total_segments: 144 | should be 144 for 6 segments.
🥣 ~ Mixing seg_idx 0 into one file
🥣 ~ Mixing seg_idx 1 into one file
🥣 ~ Mixing seg_idx 2 into one file
🥣 ~ Mixing seg_idx 3 into one file
🥣 ~ Mixing seg_idx 4 into one file
🥣 ~ Mixing seg_idx 5 into one file
🥣 ~ Mixing seg_idx 6 into one file
🥣 ~ Mixing seg_idx 7 into one file
🥣 ~ Mixing seg_idx 8 into one file
🥣 ~ Mixing seg_idx 9 into one file
🥣 ~ Mixing seg_idx 10 into one file
🥣 ~ Mixing seg_idx 11 into one file
🥣 ~ Mixing seg_idx 12 into one file
🥣 ~ Mixing seg_idx 13 into one file
🥣 ~ Mixing seg_idx 14 into one file
🥣 ~ Mixing seg_idx 15 into one file
🥣 ~ Mixing seg_idx 16 into one file
🥣 ~ Mixing seg_idx 17 into one file
🥣 ~ Mixing seg_idx 18 into one file
🥣 ~ Mixing seg_idx 19 into one file
🥣 ~ Mixing seg_idx 20 into one file
🥣 ~ Mixing seg_idx 21 into one file
🥣 ~ Mixing seg_idx 22 into one file
🥣 ~ Mixing seg_idx 23 into one file
🥣 ~ Mixing seg_idx 24 into one file
🥣 ~ Mixing seg_idx 25 into one 

## Part 5; ✨ Creating Overarching Summaries

In [ ]:
# Specify which dataset and user to process
dataset_to_process = 0  # Change this value to specify the dataset to process
user_to_process = 0     # Change this value to specify the user to process

ask_gpt_for_summary = True # Change this value to generate a summary file for the dataset and user
ask_gpt_for_entities = True # Change this value to generate the entities for the summary. (assumes a summary exists or is being created)

LLMStage = "05-superlativeSummaries/"

# Function to summarize all segments for a user
def create_overall_summary(dataset, user):
    segment_summaries = []
    for chunk in range(totalSegmentsCnt):
        print(
            f"Gathering chunk #{chunk + 1}/{totalSegmentsCnt} for final summary for user #{user} in dataset #{dataset}"
        )
        # Assuming raw_summaries contains segments for all datasets and users in order
        index = (
            (dataset * participantCnt * totalSegmentsCnt)
            + (user * totalSegmentsCnt)
            + chunk
        )
        segment_summaries.append(raw_summaries[index])

    overall_summary = summarize_all_segments(segment_summaries)
    save_json_to_file(overall_summary, LLMPartialsDirectory + LLMStage + f"ds_{dataset}_p_{str(user)}.json")
    return overall_summary


# Loop through each dataset and user
for dataset in range(0,datasetCnt):
    for user in range(0,participantCnt):
        # Check if the current dataset and user match the specified ones
        # if dataset == dataset_to_process and user == user_to_process:
        if True:
            try:
                if ask_gpt_for_summary: 
                    overall_summary = create_overall_summary(dataset, user)
                else:
                    overall_summary = load_json_file(
                        LLMPartialsDirectory
                        + LLMStage
                        + f"ds_{dataset}_p_{str(user)}.json"
                    )
                if ask_gpt_for_entities: 
                    overall_entities = identify_entities(overall_summary)
                    save_json_to_file(overall_entities, LLMPartialsDirectory
                                      + LLMStage
                                      + f"ds{dataset}_p_{str(user)}_entities.json")
                else:
                    overall_entities = load_json_file(
                        LLMPartialsDirectory
                        + LLMStage
                        + f"ds{dataset}_p_{str(user)}_entities.json"
                    )

            except Exception as e:
                # Handle the error and leave a comment
                print(f"Error processing dataset #{dataset}, user #{user}: {e}")
                # Optionally, log the error or take other actions

Now highlight the entities in each Overall summary, so we can have some terms highlighted in the overall summary.

In [ ]:
# Initialize a list to store overall summaries for each user
overall_summaries = []  

# Loop through each dataset and user
for dataset in range(datasetCnt):
    for user in range(participantCnt):
        overall_summary = load_json_file(
            LLMPartialsDirectory + LLMStage + f"ds_{dataset}_p_{str(user)}.json"
        )
        overall_entities = load_json_file(
            LLMPartialsDirectory + LLMStage + f"ds{dataset}_p_{str(user)}_entities.json"
        )
        highlighted_overall_summary = highlight_entities(
            overall_summary, overall_entities
        )
        overall_summaries.append(highlighted_overall_summary)
save_json_to_file(overall_summaries, LLMPartialsDirectory + f"LLM_superlative_summaries_{totalSegmentsCnt}.json")

## Part 6; Combining it all into one file

In [20]:
overall_summaries = load_json_file(LLMPartialsDirectory + f"LLM_superlative_summaries_{totalSegmentsCnt}.json")

# LLMStage = "06-Combine/"
universalCNTR = 0
combined_data = {}

# Adding superlatives.json to combined_data
combined_data['superlatives'] = copy.deepcopy(overall_summaries)
print("🚀 ~ combined_data:", len(combined_data["superlatives"]), type(combined_data)) # type: ignore

combined_data["segments"]={}
# for dataset in range(datasetCnt):
if True:
    # dataset = 0
    datasetSegments = load_json_file(LLMPartialsDirectory+f"Highlighted_Segment_summaries_{totalSegmentsCnt}.json")
    for seg in range(len(datasetSegments)): # type: ignore
        segmentText = datasetSegments[seg] # type: ignore
        segmentNumber = (universalCNTR)%totalSegmentsCnt
        #todo figure out the calculation for PID
        # pid = math.floor(universalCNTR/(segmentsPath+participantCnt))
        # print("🚀 ~ pid:", pid, segmentNumber, universalCNTR)
        combined_data["segments"][universalCNTR] = {
            "text" : segmentText,
            "segment" : segmentNumber,
            # "ds" : dataset,
            # "pid" : pid
            }
        universalCNTR += 1 
combined_data["segmentLength"] = universalCNTR
# Step 3: Save combined_data to a JSON file (if needed)
save_json_to_file(combined_data, parentDirectory + f"/interface/LLMManifest_{totalSegmentsCnt}-new.json")

🚀 ~ combined_data: 24 <class 'dict'>
